#task1:<br><br>

Variables: Sequence of robot positions (xi,yi) at each step<br>
Domain: Each xi,yi E {0,1,2,3,4} for a 5×5 grid<br>  
Constraints:<br>  
-Start at (1,1) and end at (4,4)<br>  
-Move only diagonally (change in x and y is +-1)<br>  
-Avoid obstacle cells<br>  


In [12]:
from ortools.sat.python import cp_model

model = cp_model.CpModel()

grid_size = 5
steps = 4   

x = [model.NewIntVar(0, grid_size-1, f'x{i}') for i in range(steps)]
y = [model.NewIntVar(0, grid_size-1, f'y{i}') for i in range(steps)]

model.Add(x[0] == 1)
model.Add(y[0] == 1)
model.Add(x[steps-1] == 4)
model.Add(y[steps-1] == 4)

for i in range(steps-1):
    dx = model.NewIntVar(-1,1,f'dx{i}')
    dy = model.NewIntVar(-1,1,f'dy{i}')

    model.Add(dx == x[i+1] - x[i])
    model.Add(dy == y[i+1] - y[i])

    model.AddMultiplicationEquality(1, [dx, dx])
    model.AddMultiplicationEquality(1, [dy, dy])

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    path = [(solver.Value(x[i]), solver.Value(y[i])) for i in range(steps)]
    print("Path:", path)
else:
    print("No solution found")

Path: [(1, 1), (2, 2), (3, 3), (4, 4)]


In [ ]:
#task2
grid = [
    [1,1,0,0,0],
    [1,1,0,1,1],
    [0,0,0,1,1],
    [0,1,1,0,0],
    [0,1,1,0,0]
]

rows, cols = 5, 5
perimeter = 0

for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 1:
            
            for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+dx, j+dy
                
                if ni < 0 or nj < 0 or ni >= rows or nj >= cols or grid[ni][nj] == 0:
                    perimeter += 1

print("Perimeter:", perimeter)

Perimeter: 24


#task3<br><br>

Variables: Order of cities visited in the tour<br>
Domain: Each position can take values from {0,1...,9}<br>
Constraints:<br>
-Each city is visited exactly once (All Different)<br>
-Tour starts and ends at the same city<br>
-Must form a complete cycle<br>


In [ ]:

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

distance_matrix = [
    [0,29,20,21,16,31,100,12,4,31],
    [29,0,15,29,28,40,72,21,29,41],
    [20,15,0,15,14,25,81,9,23,27],
    [21,29,15,0,4,12,92,12,25,13],
    [16,28,14,4,0,16,94,9,20,16],
    [31,40,25,12,16,0,95,24,36,3],
    [100,72,81,92,94,95,0,90,101,99],
    [12,21,9,12,9,24,90,0,15,25],
    [4,29,23,25,20,36,101,15,0,35],
    [31,41,27,13,16,3,99,25,35,0]
]

manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    return distance_matrix[
        manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)


search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)


solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(node)
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)

    route.append(0)

    print("Optimal Route:", route)
    print("Total Distance:", route_distance)
else:
    print("No solution found")

Optimal Route: [0, 8, 7, 2, 1, 6, 5, 9, 3, 4, 0]
Total Distance: 246
